# 02 — Feature Engineering
Demonstrates the full feature pipeline: rolling window stats, lag features, normalisation.
Validates that features are computed per-engine (no cross-engine leakage).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.loader import load_fd
from labels.rul import add_rul
from labels.binary import add_binary_label
from features.rolling import add_rolling_features
from features.lag import add_lag_features
from features.pipeline import RollingLagTransformer, FeatureScaler, get_feature_cols
from evaluation.splits import track_a_split

SENSOR_COLS = [f'sensor_{i}' for i in range(1, 22)]
print('Imports OK')

## 1. Rolling features on a single engine

In [ ]:
df = load_fd('FD001')
engine1 = df[df['unit_id'] == 1].sort_values('cycle').copy()

# Apply rolling features
engine1_feat = add_rolling_features(engine1, SENSOR_COLS, window_sizes=[10, 20, 30])

# Visualise sensor_7 raw vs rolling mean
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, w in zip(axes, [10, 20, 30]):
    ax.plot(engine1_feat['cycle'], engine1_feat['sensor_7'], alpha=0.4, label='raw', color='gray')
    ax.plot(engine1_feat['cycle'], engine1_feat[f'sensor_7_mean_w{w}'], label=f'mean_w{w}', color='steelblue')
    ax.set_title(f'sensor_7 rolling mean (w={w})')
    ax.set_xlabel('cycle')
    ax.legend()
plt.suptitle('FD001 Engine 1 — Rolling mean at different window sizes', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Degradation slope captures trend direction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ['sensor_7', 'sensor_11']):
    ax.plot(engine1_feat['cycle'], engine1_feat[f'{col}_slope_w20'],
            color='firebrick', lw=1, label='slope_w20')
    ax.axhline(0, color='black', lw=0.7, linestyle='--')
    ax.set_title(f'{col} — rolling slope (w=20)')
    ax.set_xlabel('cycle')
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Full pipeline: feature count and NaN rate

In [ ]:
transformer = RollingLagTransformer()
df_raw = add_binary_label(add_rul(load_fd('FD001')))
df_feat = transformer.transform(df_raw)

feature_cols = get_feature_cols(df_feat)
print(f'Total features: {len(feature_cols)}')
print(f'NaN rate across features: {df_feat[feature_cols].isna().mean().mean():.2%}')
print(f'\nFeature column sample:')
print(feature_cols[:12])

## 4. Normalisation — train vs test distribution

In [ ]:
ta = track_a_split('FD001')

# sensor_7_mean_w20 before and after normalisation
col = 'sensor_7_mean_w20'
if col in ta.feature_cols:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(ta.train[col].dropna(), bins=40, alpha=0.6, label='train', color='steelblue')
    axes[0].hist(ta.test[col].dropna(), bins=40, alpha=0.6, label='test', color='firebrick')
    axes[0].set_title(f'{col} — after normalisation (z-score)')
    axes[0].legend()
    axes[0].set_xlabel('Normalised value')

    # Mean very close to 0 for train; slight shift for test is intentional
    axes[1].text(0.1, 0.6, f"Train mean: {ta.train[col].mean():.4f}", transform=axes[1].transAxes, fontsize=12)
    axes[1].text(0.1, 0.5, f"Train std:  {ta.train[col].std():.4f}", transform=axes[1].transAxes, fontsize=12)
    axes[1].text(0.1, 0.35, f"Test mean: {ta.test[col].mean():.4f}", transform=axes[1].transAxes, fontsize=12)
    axes[1].text(0.1, 0.25, f"Test std:  {ta.test[col].std():.4f}", transform=axes[1].transAxes, fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'{col} not in feature_cols — feature set may differ.')

## 5. No cross-engine data leakage check

In [ ]:
# Verify that unit IDs in train and test are disjoint
train_ids = set(ta.train['unit_id'].unique())
test_ids  = set(ta.test['unit_id'].unique())
overlap = train_ids & test_ids
print('Train engine IDs (sample):', sorted(train_ids)[:5], '...')
print('Test  engine IDs (sample):', sorted(test_ids)[:5], '...')
print(f'\nEngine ID overlap: {len(overlap)} engines  ← must be 0')
assert len(overlap) == 0, 'LEAKAGE DETECTED: same engine in train and test!'